In [1]:
import json, os, re
from pathlib import Path
from tqdm import tqdm
from transformers import AutoTokenizer

# ===== CONFIG =====
INPUT = Path("processed/dataset_clean.jsonl")
OUT = Path("processed/chunks.jsonl")
TOKENIZER = "sentence-transformers/all-MiniLM-L6-v2"
MAX_TOKENS = 450
OVERLAP = 100
# ==================

def sent_split(text):
    text = re.sub(r'\s+', ' ', text).strip()
    parts = re.split(r'(?<=[.!?…])\s+|\n+', text)
    return [p.strip() for p in parts if p.strip()]

def chunk_text(sents, tokenizer):
    chunks, cur, cur_tokens = [], [], 0
    for s in sents:
        toks = len(tokenizer.tokenize(s))
        if cur_tokens + toks > MAX_TOKENS:
            chunks.append(" ".join(cur).strip())
            new_cur, new_tokens = [], 0
            for sent in reversed(cur):
                t = len(tokenizer.tokenize(sent))
                if new_tokens + t > OVERLAP: break
                new_cur.insert(0, sent)
                new_tokens += t
            cur, cur_tokens = new_cur, new_tokens
        cur.append(s)
        cur_tokens += toks
    if cur: chunks.append(" ".join(cur).strip())
    return chunks

tokenizer = AutoTokenizer.from_pretrained(TOKENIZER, use_fast=True)
os.makedirs(OUT.parent, exist_ok=True)

with open(INPUT, "r", encoding="utf-8") as fin, open(OUT, "w", encoding="utf-8") as fout:
    for line in tqdm(fin, desc="Chunking docs"):
        doc = json.loads(line)
        text = (doc.get("content") or doc.get("title") or "").strip()
        if not text: continue
        sents = sent_split(text)
        chunks = chunk_text(sents, tokenizer)
        for i, ch in enumerate(chunks):
            obj = {
                "id": f"{doc['id']}_chunk_{i+1}",
                "source_id": doc["id"],
                "domain": doc.get("domain", ""),
                "title": doc.get("title", ""),
                "source_url": doc.get("source_url", ""),
                "chunk_index": i,
                "text": ch,
                "tokens_est": len(tokenizer.tokenize(ch))
            }
            fout.write(json.dumps(obj, ensure_ascii=False) + "\n")
print("Done:", OUT)


c:\Users\tuand\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Disabling PyTorch because PyTorch >= 2.1 is required but found 2.0.1+cu117
c:\Users\tuand\AppData\Local\Programs\Python\Python310\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\tuand\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To suppo

Done: processed\chunks.jsonl
